In [1]:
# INSTALL
!pip install geemap geopandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 46.9 MB/s eta 0:00:00


In [2]:
# IMPORT
import ee
import geemap
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from google.colab import drive

In [3]:
# MOUNT DRIVE & AUTH
drive.mount('/content/drive')
ee.Authenticate()
ee.Initialize(project='ee-defaniarman')

Mounted at /content/drive


In [4]:
# LOAD SHAPEFILE
shp_path = '/content/drive/MyDrive/sampel_lapangan.shp'
gdf = gpd.read_file(shp_path)
gdf = gdf.to_crs(epsg=4326)
print(f'{len(gdf)} titik | CRS: {gdf.crs}')
gdf.head()

50 titik | CRS: EPSG:4326


,rand_point,id,geometry
0,0.0,1,POINT (109.07393 -6.79374)
1,1.0,1,POINT (109.07714 -6.79703)
2,2.0,1,POINT (109.07818 -6.78581)
3,3.0,1,POINT (109.0751 -6.78546)
4,4.0,1,POINT (109.07186 -6.79231)


In [5]:
# CONVERT TO EE FEATURECOLLECTION
fc_samples = geemap.gdf_to_ee(gdf)

In [6]:
# AOI + S2 COMPOSITE
aoi = ee.FeatureCollection('projects/ee-defaniarman/assets/aoi_crb').geometry()

csPlus = ee.ImageCollection('GOOGLE/CLOUD_SCORE_PLUS/V1/S2_HARMONIZED')

def maskS2CSPlus(image):
    qa = image.select('cs_cdf')
    return image.updateMask(qa.gte(0.60))

s2_base = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
    .filterBounds(aoi)
    .filterDate('2024-01-01', '2024-12-31')
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)))

composite = (s2_base
    .linkCollection(csPlus, ['cs_cdf'])
    .map(maskS2CSPlus)
    .median()
    .clip(aoi)
    .divide(10000))

print('Composite bands:', composite.bandNames().getInfo())

Composite bands: ['B1', 'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B9', 'B11', 'B12', 'AOT', 'WVP', 'SCL', 'TCI_R', 'TCI_G', 'TCI_B', 'MSK_CLDPRB', 'MSK_SNWPRB', 'QA10', 'QA20', 'QA60', 'MSK_CLASSI_OPAQUE', 'MSK_CLASSI_CIRRUS', 'MSK_CLASSI_SNOW_ICE', 'cs_cdf']


In [7]:
# HITUNG INDEKS
B3  = composite.select('B3')   # Green  560 nm
B4  = composite.select('B4')   # Red    665 nm
B8  = composite.select('B8')   # NIR    842 nm
B11 = composite.select('B11')  # SWIR1 1610 nm

NDVI = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
NDWI = composite.normalizedDifference(['B3', 'B8']).rename('NDWI')
NDMI = composite.normalizedDifference(['B8', 'B11']).rename('NDMI')
NDBI = composite.normalizedDifference(['B11', 'B8']).rename('NDBI')

BANDS_SPECTRAL = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12']
img_full = (composite.select(BANDS_SPECTRAL)
    .addBands([NDVI, NDWI, NDMI, NDBI]))

print('Full image bands:', img_full.bandNames().getInfo())

Full image bands: ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'NDMI', 'NDBI']


In [8]:
# SAMPLING
sampled = img_full.sampleRegions(
    collection = fc_samples,
    properties = ['rand_point', 'id'],
    scale      = 10,
    geometries = True
)

df_sample = geemap.ee_to_df(sampled)
print(df_sample.shape)
df_sample.head()

(50, 12)


,B11,B12,B2,B3,B4,B8,NDBI,NDMI,NDVI,NDWI,id,rand_point
0,0.1443,0.0731,0.0650,0.0886,0.0928,0.1320,0.044517,-0.044517,0.174377,-0.196736,1,0
1,0.1332,0.0691,0.0378,0.0593,0.0448,0.2160,-0.237113,0.237113,0.656442,-0.569197,1,1
2,0.1216,0.0553,0.0407,0.0756,0.0464,0.3124,-0.439631,0.439631,0.741360,-0.610309,1,2
3,0.1324,0.0590,0.0355,0.0611,0.0384,0.2888,-0.371320,0.371320,0.765281,-0.650757,1,3
4,0.1241,0.0493,0.0374,0.0661,0.0377,0.3364,-0.461021,0.461021,0.798450,-0.671553,1,4


In [9]:
# TAMBAH KOORDINAT
coords = gdf[['geometry']].copy()
coords['lon'] = gdf.geometry.x
coords['lat'] = gdf.geometry.y
coords['rand_point'] = gdf['rand_point']

df_out = df_sample.merge(
    coords[['rand_point', 'lon', 'lat']],
    on='rand_point', how='left'
)

col_order = ['rand_point', 'id', 'lon', 'lat',
             'B2', 'B3', 'B4', 'B8', 'B11', 'B12',
             'NDVI', 'NDWI', 'NDMI', 'NDBI']
df_out = df_out[[c for c in col_order if c in df_out.columns]]
df_out = df_out.sort_values('rand_point').reset_index(drop=True)
print(df_out.shape)
df_out.head()

(50, 14)


,rand_point,id,lon,lat,B2,B3,B4,B8,B11,B12,NDVI,NDWI,NDMI,NDBI
0,0,1,109.073931,-6.793742,0.0650,0.0886,0.0928,0.1320,0.1443,0.0731,0.174377,-0.196736,-0.044517,0.044517
1,1,1,109.077139,-6.797030,0.0378,0.0593,0.0448,0.2160,0.1332,0.0691,0.656442,-0.569197,0.237113,-0.237113
2,2,1,109.078181,-6.785813,0.0407,0.0756,0.0464,0.3124,0.1216,0.0553,0.741360,-0.610309,0.439631,-0.439631
3,3,1,109.075095,-6.785461,0.0355,0.0611,0.0384,0.2888,0.1324,0.0590,0.765281,-0.650757,0.371320,-0.371320
4,4,1,109.071862,-6.792308,0.0374,0.0661,0.0377,0.3364,0.1241,0.0493,0.798450,-0.671553,0.461021,-0.461021


In [10]:
# EXPORT CSV + GPKG
out_dir = '/content/drive/MyDrive/'

csv_path  = out_dir + 'sampel_indeks.csv'
gpkg_path = out_dir + 'sampel_indeks.gpkg'

df_out.to_csv(csv_path, index=False)

gdf_out = gpd.GeoDataFrame(
    df_out,
    geometry=gpd.points_from_xy(df_out['lon'], df_out['lat']),
    crs='EPSG:4326'
)
gdf_out.to_file(gpkg_path, driver='GPKG', layer='sampel_indeks')

print('CSV  ->', csv_path)
print('GPKG ->', gpkg_path)

CSV  -> /content/drive/MyDrive/sampel_indeks.csv
GPKG -> /content/drive/MyDrive/sampel_indeks.gpkg
